# 1. Import
We need to import some libraries

In [2]:
import requests
from tqdm.notebook import tqdm
import zipfile
import os
import pandas as pd
from pyproj import Transformer
import sys
import traci

Then we need to download dataset:

In [3]:
url = "https://data.transportation.gov/api/views/8ect-6jqj/files/10ed9f47-ab08-4df2-b46a-29f1683ceffa?download=true&filename=I-80-Emeryville-CA.zip"
filename = 'I-80-Emeryville-CA.zip'
response = requests.get(url, stream=True)
response.raise_for_status()
total_size = int(response.headers.get('content-length', 0))
chunk_size = 1024 * 1024
with open(filename, 'wb') as file, tqdm(
    desc=filename,
    total=total_size,
    unit='iB',
    unit_scale=True,
    unit_divisor=1024,
    colour='#00ff00'
) as bar:
    for data in response.iter_content(chunk_size=chunk_size):
        size = file.write(data)
        bar.update(size)
data_folder = 'data'
with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall(data_folder)
    print(filename, " is extracted to ", data_folder, " folder.")

I-80-Emeryville-CA.zip: 0.00iB [00:00, ?iB/s]

I-80-Emeryville-CA.zip  is extracted to  data  folder.


Extract it:

In [4]:
for file in os.listdir(data_folder):
    if file.endswith(".zip"):
        zip_path = os.path.join(data_folder, file)
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(data_folder)

            os.remove(zip_path)
            print("Done:", file)
        except:
            print("Error with:", file)
os.remove(filename)
print("all done")

Done: i-80-aerial-ortho-photos.zip
Done: i-80-cad-diagram.zip
Done: i-80-data-analysis.zip
Done: i-80-detector-data.zip
Done: i-80-gis-files.zip
Done: i-80-signal-timing.zip
Done: i-80-signs.zip
Done: i-80-vehicle-trajectory-data.zip
Done: i-80-weather-data.zip
all done


# 2. To python readable data

We need to get our data into pandas dataframe format:

In [6]:
pd.set_option('display.float_format', '{:.3f}'.format)
df1 = pd.read_csv("data/vehicle-trajectory-data/0400pm-0415pm/trajectories-0400-0415.csv")
df2=pd.read_csv("data/vehicle-trajectory-data/0500pm-0515pm/trajectories-0500-0515.csv")
df3=pd.read_csv("data/vehicle-trajectory-data/0515pm-0530pm/trajectories-0515-0530.csv")
dfs = [df1,df2,df3]
df=pd.concat(dfs, axis=0, ignore_index=True)
df.head()

,Vehicle_ID,Frame_ID,Total_Frames,Global_Time,Local_X,Local_Y,Global_X,Global_Y,v_Length,v_Width,v_Class,v_Vel,v_Acc,Lane_ID,Preceding,Following,Space_Headway,Time_Headway
0,1,12,884,1113433136100,16.884,48.213,6042842.116,2133117.662,14.300,6.400,2,12.500,0.000,2,0,0,0.000,0.000
1,1,13,884,1113433136200,16.938,49.463,6042842.012,2133118.909,14.300,6.400,2,12.500,0.000,2,0,0,0.000,0.000
2,1,14,884,1113433136300,16.991,50.712,6042841.908,2133120.155,14.300,6.400,2,12.500,0.000,2,0,0,0.000,0.000
3,1,15,884,1113433136400,17.045,51.963,6042841.805,2133121.402,14.300,6.400,2,12.500,0.000,2,0,0,0.000,0.000
4,1,16,884,1113433136500,17.098,53.213,6042841.701,2133122.649,14.300,6.400,2,12.500,0.000,2,0,0,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145720,1757,6959,1386,1113438260800,29.320,655.411,6042779.679,2133721.623,30.300,8.500,3,0.000,0.000,3,1744,1764,158.900,9999.990
3145721,1757,6960,1386,1113438260900,29.320,655.411,6042779.679,2133721.623,30.300,8.500,3,0.000,0.000,3,1744,1764,159.680,9999.990
3145722,1757,6961,1386,1113438261000,29.320,655.411,6042779.679,2133721.623,30.300,8.500,3,0.000,0.000,3,1744,1764,160.650,9999.990
3145723,1757,6962,1386,1113438261100,29.320,655.411,6042779.679,2133721.623,30.300,8.500,3,0.000,0.000,3,1744,1764,161.680,9999.990


# 3. Map downloading

In [24]:
us_to_sumo = Transformer.from_crs("EPSG:2227", "EPSG:32610", always_xy=True)  # NGSIM data projection to SUMO data projection convertion function
# EPSG:2227 is given in I-80 Metadata Documentation.pdf as "NAD83 – California State Plane Coordinate System, Zone 3". EPSG:32610 is given in map.net.xml file inside <location> tag as projParameter="+proj=utm +zone=10 +ellps=WGS84 +datum=WGS84 +units=m +no_defs"
gb = {'e':float(df["Global_X"].max()), 'w':float(df["Global_X"].min()), 'n':float(df["Global_Y"].max()), 's':float(df["Global_Y"].min())}
rectangle = 2000
ew = gb['e']-gb['w']
ns = gb['n']-gb['s']
ewad = (rectangle - ew) / 2
nsad = (rectangle - ns) / 2
# ad = (rectangle - m) / 2
# ad = float(df['v_Length'].max()) * 2
# print(ad)
gb['e'] = gb['e'] + ewad
gb['w'] = gb['w'] - ewad
gb['n'] = gb['n'] + nsad
gb['s'] = gb['s'] - nsad

In [ ]:
We need to dowload OpenStreetMap:

In [25]:
to_latLong = Transformer.from_crs("EPSG:2227", "EPSG:4326", always_xy=True)
(east, north) = to_latLong.transform(gb['e'],gb['n'])
(west, south) = to_latLong.transform(gb['w'],gb['s'])
url=f"https://api.openstreetmap.org/api/0.6/map?bbox={west},{south},{east},{north}"
print(url)
filename = "map.osm"
headers = {
    'User-Agent': 'MyOSMDownloader/1.0 (kamolovmaxmud@gmail.com)'
}
response = requests.get(url, stream=True, headers=headers)
with open(filename, 'wb') as file, tqdm(
    desc=filename,
    total=total_size,
    unit='iB',
    unit_scale=True,
    unit_divisor=1024,
    colour='#00ff00'
) as bar:
    for data in response.iter_content(chunk_size=chunk_size):
        size = file.write(data)
        bar.update(size)
print(f"Downloaded the file: {filename}")
!netconvert --osm-files map.osm -o map.net.xml --no-warnings t
print("converting is done: map.net.xml")

https://api.openstreetmap.org/api/0.6/map?bbox=-122.30055486352956,37.83895674069398,-122.29376301151724,37.84455380605861


map.osm: 0.00iB [00:00, ?iB/s]

Downloaded the file: map.osm
Success.
converting is done: map.net.xml


In [26]:
(east, north) = us_to_sumo.transform(gb['e'],gb['n'])
print("east = ",east, " north = ", north)
(west, south) = us_to_sumo.transform(gb['w'],gb['s'])
print("west = ",west, " south = ", south)

east =  562136.7680777116  north =  4188803.1272444665
west =  561543.8449000294  south =  4188177.628821053


After getting east west north south coordinates in EPSG:32610, I used these in QGIS to get images.I installed NextGIS QuickMapServices plugin to QGIS. On toolbar section, QuickMapServices -> Google -> Google satellite. I added these coordinates to a bookmark. I set QGIS projection to EPSG:32610 and then Project -> Import/Export -> Export Map to Image... -> scale 1:200, Resolution 96 dpi, Append Georeference Information (embedded or via world file)✅

I got the image and .pgw extension world file. I used image pixel resolution and all the parameters on world file.

In [29]:
netOffsetX = -561498.24 # it came from netOffset in 27th line of map.net.xml
netOffsetY = -4188064.33 # it came from netOffset in 27th line of map.net.xml
px_1 = 14005 # image pixel resolution
px_2 = 14756 # image pixel resolution
x_scale = 0.04238871145537474 # 1st line of .pgw file
y_scale = -0.04238871145537474 # 4th line of .pgw file
left_boundary = 561543.50602984894067049 # 5th line of .pgw file
left_boundary = left_boundary + netOffsetX
top_boundary = 4188803.10075152199715376 # 6th line of .pgw file
top_boundary = top_boundary + netOffsetY
height = px_2*y_scale
width = px_1*x_scale
top_left = [top_boundary + 0.5 * x_scale, left_boundary - 0.5 * x_scale]
bottom_down = [top_boundary + 0.5 * x_scale + height, left_boundary - 0.5 * x_scale + width]
centerY = (top_left[0] + bottom_down[0])/2
centerX = (top_left[1] + bottom_down[1])/2
xml_content = f"""<viewsettings>
    <scheme name="real world"/>
    <decal file="img.png" centerX="{centerX}" centerY="{centerY}" width="{width}" height="{abs(height)}" rotation="0" layer="20"/>
</viewsettings>"""
file_path = "viewsettings.xml"
with open(file_path, "w") as file:
    file.write(xml_content)

# 4. Data manipulation:

In [30]:
min_time = df["Global_Time"].min() # first recorded data time
sorted_df=df.sort_values(by="Global_Time") # sorting data according to time
sorted_df["Global_Time"]=sorted_df["Global_Time"]-min_time # making simulation start time to 0.0
sorted_df["Global_Time"]=sorted_df["Global_Time"]/1000 # milliseconds to seconds
sorted_df["Local_X"]=sorted_df["Local_X"]*0.3048 # feet2meter conversion
sorted_df["Local_Y"]=sorted_df["Local_Y"]*0.3048 # feet2meter conversion
sorted_df["v_Length"]=sorted_df["v_Length"]*0.3048 # feet2meter conversion
sorted_df["v_Width"]=sorted_df["v_Width"]*0.3048 # feet2meter conversion
sorted_df["v_Vel"]=sorted_df["v_Vel"]*0.3048 # feet2meter conversion
sorted_df["v_Acc"]=sorted_df["v_Acc"]*0.3048 # feet2meter conversion
sorted_df["Space_Headway"]=sorted_df["Space_Headway"]*0.3048 # feet2meter conversion
(sorted_df["Local_X"],sorted_df["Local_Y"])=us_to_sumo.transform(sorted_df["Global_X"],sorted_df["Global_Y"]) # NGSIM data projection to SUMO data projection convertion
sorted_df["Local_X"]=sorted_df["Local_X"] + netOffsetX
sorted_df["Local_Y"]=sorted_df["Local_Y"] + netOffsetY
grouped_by_time=sorted_df.groupby("Global_Time") # for getting all each timeframe one by one
timeframes = sorted(grouped_by_time.groups.keys()) # for iteration
sorted_df.head() # to see the data appearance

,Vehicle_ID,Frame_ID,Total_Frames,Global_Time,Local_X,Local_Y,Global_X,Global_Y,v_Length,v_Width,v_Class,v_Vel,v_Acc,Lane_ID,Preceding,Following,Space_Headway,Time_Headway
11027,36,4,1005,0.000,379.352,182.951,6042848.877,2133146.021,7.559,2.591,3,3.377,0.000,3,0,0,0.000,0.000
11028,36,5,1005,0.100,379.306,183.253,6042848.754,2133147.014,7.559,2.591,3,3.377,0.000,3,0,0,0.000,0.000
11029,36,6,1005,0.200,379.261,183.554,6042848.631,2133148.006,7.559,2.591,3,3.377,0.000,3,0,0,0.000,0.000
11030,36,7,1005,0.300,379.238,183.704,6042848.570,2133148.502,7.559,2.591,3,3.377,0.000,3,0,0,0.000,0.000
11031,36,8,1005,0.400,379.252,184.631,6042848.696,2133151.541,7.559,2.591,3,3.377,0.000,3,0,0,0.000,0.000


# 5. Running on SUMO:

In [31]:
if 'SUMO_HOME' in os.environ:
    sys.path.append(os.path.join(os.environ['SUMO_HOME'], 'tools'))


SUMO_CMD = "sumo-gui"
NET_FILE = "map.net.xml"


vehicles = {}
vehicle_types = ["DEFAULT_VEHTYPE", "motorcycle", "passenger", "truck"]
last_frame_vehicles = []



try:
    traci.start([SUMO_CMD, "-n", NET_FILE, "--gui-settings-file", "viewsettings.xml",  '--delay', '100'])
    for time in timeframes:
        current_frame = grouped_by_time.get_group(time)
        for index, row in current_frame.iterrows():
    
            ########################################################################################
            # initialisation
            id = int(row["Vehicle_ID"])
            fid = int(row["Frame_ID"])
            tf = int(row["Total_Frames"])
            x = float(row["Local_X"])
            y = float(row["Local_Y"])
            len = float(row["v_Length"])
            wid = float(row["v_Width"])
            cl = int(row["v_Class"])
            vel = float(row["v_Vel"])
            ac = float(row["v_Acc"])
            lid = int(row["Lane_ID"])
            pre = int(row["Preceding"])
            fol = int(row["Following"])
            shead = float(row["Space_Headway"])
            thead = float(row["Time_Headway"])
            ########################################################################################
    
            if not id in vehicles.keys():
                ##################################################################################
                traci.vehicle.add(
                    vehID=id,
                    routeID="",
                    )
                traci.vehicle.setVehicleClass(
                    typeID=id,
                    clazz=vehicle_types[cl]
                )
                traci.vehicle.setShapeClass(
                    typeID=id,
                    shapeClass=vehicle_types[cl]
                    )
                traci.vehicle.setLength(
                    typeID=id,
                    length=len
                    )
                traci.vehicle.setWidth(
                    typeID=id,
                    width=wid
                    )
                vehicles[id]=0
                ##################################################################################
            
            traci.vehicle.moveToXY(
                    vehID=id,
                    edgeID="",
                    laneIndex=lid,
                    x=x,
                    y=y,
                    keepRoute=2
                    )
            vehicles[id]=vehicles[id]+1
            if vehicles[id] == tf:
                last_frame_vehicles.append(id)
            
            
        traci.simulationStep()
        if last_frame_vehicles is not None:
            for i in last_frame_vehicles:
                traci.vehicle.remove(i)
                last_frame_vehicles.remove(i)
except Exception as e:
    print("ERROR: ", e)
finally:
    traci.close()


 Retrying in 1 seconds
ERROR:  Connection closed by SUMO.
